# Notebook 03 — Detection Methods

In Notebook 02 you could *see* the changepoint just by eyeballing the bit-flip
rate plot.  Real-world quantum hardware runs continuously — you cannot pause
the experiment and stare at a graph every few timesteps.  You need an
**algorithm** that watches the signal and raises an alarm automatically.

This notebook builds and compares three classical statistical detectors:

| Method | One-line intuition |
|--------|--------------------|
| **CUSUM** | Accumulate evidence of an upward shift; alarm when the bucket overflows. |
| **BOCD**  | At every step, compute the Bayesian probability that a changepoint *just* happened. |
| **Window**| Compare a "past" window to a "recent" window using KL divergence; alarm when they diverge. |

At the end we compare all three: which detects the fastest? Which is most
robust to noise? How does the choice of threshold affect precision vs. speed?

**Connection to the PhD research**
- *Giarmatzi / Tonekaboni*: Before you can characterise *how* noise is correlated
  over time, you first need to know *when* the noise regime changed.  These
  detectors provide that segmentation.
- *Sanders / Usman*: If the device drifts mid-experiment, your verification
  conclusions mix two noise regimes and become unreliable.  A fast detector
  tells you which data to trust.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from quantum_drift_detector.simulators import DepolarizingSimulator
from quantum_drift_detector.features import extract_bit_flip_rate
from quantum_drift_detector.detectors import run_cusum, run_bocd, run_window_detector

# Reproducibility
rng = np.random.default_rng(42)
np.random.seed(42)

## 1 — Recreate the Phase 1 dataset

We use the same setup as Notebook 02: 100 timesteps, 1 000 shots each,
with the error rate jumping from 1 % to 5 % at timestep 50.
This is a moderate shift — not obvious at every individual timestep,
but visible in the smoothed bit-flip rate.

In [ ]:
TRUE_CHANGEPOINT = 50
N_TIMESTEPS      = 100
N_SHOTS          = 1_000
ERROR_RATE_PRE   = 0.01   # 1 %  before the change
ERROR_RATE_POST  = 0.05   # 5 %  after the change

sim = DepolarizingSimulator(
    error_rate_pre=ERROR_RATE_PRE,
    error_rate_post=ERROR_RATE_POST,
    changepoint=TRUE_CHANGEPOINT,
)
data = sim.generate_data(n_timesteps=N_TIMESTEPS, n_shots=N_SHOTS)
bit_flip_rates = extract_bit_flip_rate(data)

print(f"Data shape           : {data.shape}")
print(f"Mean rate pre-change : {bit_flip_rates[:TRUE_CHANGEPOINT].mean():.4f}")
print(f"Mean rate post-change: {bit_flip_rates[TRUE_CHANGEPOINT:].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(bit_flip_rates, color='steelblue', alpha=0.8, label='Bit-flip rate')
ax.axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', label=f'True changepoint (t={TRUE_CHANGEPOINT})')
ax.set(xlabel='Timestep', ylabel='Bit-flip rate', title='Phase 1 data — raw bit-flip rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2 — Detector 1: CUSUM

### How it works (plain English)

Think of CUSUM as a rain gauge.  Every timestep you add the amount by which
the current bit-flip rate *exceeds* what you expect (the baseline + a small
slack).  If the rate is normal or low, the gauge empties (resets to zero).
When something breaks and the rate consistently runs high, the gauge fills up
until it overflows — that overflow is your alarm.

Two parameters to tune:
- **allowance** (`k`): how much above the baseline to ignore.  Half the
  expected shift size is the standard choice.
- **threshold** (`h`): how full the gauge must get before you raise an alarm.
  Higher = fewer false alarms, slower detection.

### Mathematical formula

$$S_t = \max\!\left(0,\; S_{t-1} + (x_t - \mu_0 - k)\right), \qquad \text{alarm when } S_t > h$$

In [ ]:
# Run CUSUM.
# target_mean = baseline error rate (what we expect before any change)
# allowance   = half the shift we expect: (0.05 - 0.01) / 2 = 0.02
# threshold   = tuned by inspection; 0.05 works well here

cusum_result = run_cusum(
    bit_flip_rates,
    target_mean=ERROR_RATE_PRE,
    allowance=0.02,
    threshold=0.05,
)

print(f"CUSUM detected changepoint at: t = {cusum_result['detected_at']}")
print(f"True changepoint was at:       t = {TRUE_CHANGEPOINT}")
if cusum_result['detected_at'] is not None:
    delay = cusum_result['detected_at'] - TRUE_CHANGEPOINT
    print(f"Detection delay: {delay} timesteps")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(bit_flip_rates, color='steelblue', alpha=0.8)
axes[0].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', label='True changepoint')
if cusum_result['detected_at']:
    axes[0].axvline(cusum_result['detected_at'], color='green', linestyle=':', label='CUSUM alarm')
axes[0].set(ylabel='Bit-flip rate', title='CUSUM — raw signal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(cusum_result['scores'], color='darkorange', label='CUSUM score $S_t$')
axes[1].axhline(cusum_result['threshold'], color='purple', linestyle='--', label=f"Threshold h = {cusum_result['threshold']}")
axes[1].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--')
if cusum_result['detected_at']:
    axes[1].axvline(cusum_result['detected_at'], color='green', linestyle=':')
axes[1].set(xlabel='Timestep', ylabel='CUSUM score', title='CUSUM score over time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3 — Detector 2: Bayesian Online Change-Point Detection (BOCD)

### How it works (plain English)

BOCD maintains a *belief distribution* over how long the current regime has
been running — the "run length" r.  At every timestep it asks:

> "Given everything I've seen, what's the probability that a changepoint
> literally just happened (run length = 0)?"

Using Bayes' rule, when the new observation looks very different from what
the recent run would predict, probability mass shifts to r = 0.
That shifting probability is your alarm signal.

The observation model here is **Normal likelihood with unknown mean and
variance**, with a Normal-InvGamma conjugate prior.  The predictive
distribution (what BOCD expects at the next step given run length r)
is a Student-*t* distribution — wider and heavier-tailed than Normal,
reflecting uncertainty about the true variance.

**Key prior parameters to set:**
- `mu_0`: your prior guess for the baseline bit-flip rate.
- `hazard_rate`: prior probability of a changepoint at each step
  (= 1 / expected segment length).

In [ ]:
# hazard_rate = 1/100 means we expect ~1 changepoint per 100 timesteps.
# mu_0 = our prior guess for the pre-change bit-flip rate.

bocd_result = run_bocd(
    bit_flip_rates,
    hazard_rate=0.01,
    mu_0=ERROR_RATE_PRE,
    threshold=0.4,
)

print(f"BOCD detected changepoint at: t = {bocd_result['detected_at']}")
print(f"True changepoint was at:      t = {TRUE_CHANGEPOINT}")
if bocd_result['detected_at'] is not None:
    delay = bocd_result['detected_at'] - TRUE_CHANGEPOINT
    print(f"Detection delay: {delay} timesteps")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(bit_flip_rates, color='steelblue', alpha=0.8)
axes[0].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', label='True changepoint')
if bocd_result['detected_at']:
    axes[0].axvline(bocd_result['detected_at'], color='green', linestyle=':', label='BOCD alarm')
axes[0].set(ylabel='Bit-flip rate', title='BOCD — raw signal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(bocd_result['changepoint_probs'], color='darkorange',
             label='P(changepoint at t)')
axes[1].axhline(bocd_result['threshold'], color='purple', linestyle='--',
                label=f"Threshold = {bocd_result['threshold']}")
axes[1].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--')
if bocd_result['detected_at']:
    axes[1].axvline(bocd_result['detected_at'], color='green', linestyle=':')
axes[1].set(xlabel='Timestep', ylabel='P(changepoint)',
            title='Bayesian changepoint probability over time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4 — Detector 3: Sliding-Window KL Divergence

### How it works (plain English)

Slide two adjacent windows of equal width along the time series.  At each
position, estimate the bit-flip probability in the "past" window and the
"recent" window.  Then ask: *how different are these two distributions?*

We measure the difference with **KL divergence** — an information-theoretic
measure of surprise.  KL divergence between two Bernoulli distributions
Bernoulli(*p*) and Bernoulli(*q*) has a simple closed form:

$$\mathrm{KL}(p \| q) = p \log\frac{p}{q} + (1-p)\log\frac{1-p}{1-q}$$

We use the *symmetric* version (sum both directions) to avoid directional
bias.  When the two windows are identical, KL = 0.  When the rate has
shifted, KL spikes.

**Trade-off with window size:**
- Larger window → more stable estimates, slower detection.
- Smaller window → faster detection, noisier scores.

In [ ]:
window_result = run_window_detector(
    bit_flip_rates,
    window_size=10,
    threshold=0.05,
)

print(f"Window detector detected changepoint at: t = {window_result['detected_at']}")
print(f"True changepoint was at:                 t = {TRUE_CHANGEPOINT}")
if window_result['detected_at'] is not None:
    delay = window_result['detected_at'] - TRUE_CHANGEPOINT
    print(f"Detection delay: {delay} timesteps")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(bit_flip_rates, color='steelblue', alpha=0.8)
axes[0].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', label='True changepoint')
if window_result['detected_at']:
    axes[0].axvline(window_result['detected_at'], color='green', linestyle=':', label='Window alarm')
axes[0].set(ylabel='Bit-flip rate', title='Sliding-window KL detector — raw signal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(window_result['scores'], color='darkorange', label='Symmetric KL score')
axes[1].axhline(window_result['threshold'], color='purple', linestyle='--',
                label=f"Threshold = {window_result['threshold']}")
axes[1].axvline(TRUE_CHANGEPOINT, color='red', linestyle='--')
if window_result['detected_at']:
    axes[1].axvline(window_result['detected_at'], color='green', linestyle=':')
axes[1].set(xlabel='Timestep', ylabel='KL divergence',
            title='Symmetric KL divergence between past and recent windows')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5 — Side-by-side comparison

Let's put all three detectors on one figure to compare them at a glance.

In [ ]:
fig = plt.figure(figsize=(13, 9))
gs  = gridspec.GridSpec(4, 1, hspace=0.45)

t = np.arange(N_TIMESTEPS)

# ── Panel 0: raw signal ──────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
ax0.plot(t, bit_flip_rates, color='steelblue', alpha=0.8)
ax0.axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', lw=1.5)
ax0.set(ylabel='Bit-flip rate', title='Raw signal')
ax0.grid(True, alpha=0.3)

# ── Panel 1: CUSUM ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax1.plot(t, cusum_result['scores'], color='darkorange')
ax1.axhline(cusum_result['threshold'], color='purple', linestyle='--', lw=1)
ax1.axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', lw=1.5)
if cusum_result['detected_at']:
    ax1.axvline(cusum_result['detected_at'], color='green', linestyle=':', lw=2,
                label=f"CUSUM alarm t={cusum_result['detected_at']}")
    ax1.legend(fontsize=9)
ax1.set(ylabel='Score', title='CUSUM')
ax1.grid(True, alpha=0.3)

# ── Panel 2: BOCD ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[2], sharex=ax0)
ax2.plot(t, bocd_result['changepoint_probs'], color='teal')
ax2.axhline(bocd_result['threshold'], color='purple', linestyle='--', lw=1)
ax2.axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', lw=1.5)
if bocd_result['detected_at']:
    ax2.axvline(bocd_result['detected_at'], color='green', linestyle=':', lw=2,
                label=f"BOCD alarm t={bocd_result['detected_at']}")
    ax2.legend(fontsize=9)
ax2.set(ylabel='P(change)', title='Bayesian BOCD')
ax2.grid(True, alpha=0.3)

# ── Panel 3: Window KL ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[3], sharex=ax0)
ax3.plot(t, window_result['scores'], color='crimson')
ax3.axhline(window_result['threshold'], color='purple', linestyle='--', lw=1)
ax3.axvline(TRUE_CHANGEPOINT, color='red', linestyle='--', lw=1.5)
if window_result['detected_at']:
    ax3.axvline(window_result['detected_at'], color='green', linestyle=':', lw=2,
                label=f"Window alarm t={window_result['detected_at']}")
    ax3.legend(fontsize=9)
ax3.set(xlabel='Timestep', ylabel='KL div.', title='Sliding Window (KL)')
ax3.grid(True, alpha=0.3)

plt.suptitle('Three detectors on the same dataset', fontsize=13, y=1.01)
plt.show()

# ── Summary table ────────────────────────────────────────────────────────
print("\n--- Detection summary ---")
print(f"True changepoint : t = {TRUE_CHANGEPOINT}")
for name, det in [("CUSUM ", cusum_result['detected_at']),
                  ("BOCD  ", bocd_result['detected_at']),
                  ("Window", window_result['detected_at'])]:
    if det is not None:
        print(f"{name}          : t = {det}  (delay = {det - TRUE_CHANGEPOINT:+d})")
    else:
        print(f"{name}          : no detection")

## 6 — Threshold sensitivity analysis

In your stats training you have seen the precision-recall / ROC trade-off.
Here the equivalent is: **detection delay vs. false-alarm rate**.

- Low threshold → detect earlier (low delay), but also alarm on noise (more
  false positives).
- High threshold → miss fewer noise-driven alarms, but react later to real
  changes.

We sweep the threshold for CUSUM over many random realisations of the same
scenario and compute the **average detection delay** and the
**false-alarm rate** (fraction of runs with no true change where an alarm
fires anyway).

In [ ]:
N_TRIALS   = 200
thresholds = np.linspace(0.01, 0.30, 30)

avg_delays    = []
false_alarm_rates = []

for h in thresholds:
    delays      = []
    false_alarms = 0

    for _ in range(N_TRIALS):
        # Scenario A: real changepoint at t=50
        sim_cp = DepolarizingSimulator(ERROR_RATE_PRE, ERROR_RATE_POST, TRUE_CHANGEPOINT)
        rates_cp = extract_bit_flip_rate(sim_cp.generate_data(N_TIMESTEPS, N_SHOTS))
        res = run_cusum(rates_cp, target_mean=ERROR_RATE_PRE, allowance=0.02, threshold=h)
        if res['detected_at'] is not None and res['detected_at'] >= TRUE_CHANGEPOINT:
            delays.append(res['detected_at'] - TRUE_CHANGEPOINT)

        # Scenario B: no changepoint (constant noise)
        sim_flat = DepolarizingSimulator(ERROR_RATE_PRE, ERROR_RATE_PRE, TRUE_CHANGEPOINT)
        rates_flat = extract_bit_flip_rate(sim_flat.generate_data(N_TIMESTEPS, N_SHOTS))
        res_flat = run_cusum(rates_flat, target_mean=ERROR_RATE_PRE, allowance=0.02, threshold=h)
        if res_flat['detected_at'] is not None:
            false_alarms += 1

    avg_delays.append(np.mean(delays) if delays else np.nan)
    false_alarm_rates.append(false_alarms / N_TRIALS)

avg_delays        = np.array(avg_delays)
false_alarm_rates = np.array(false_alarm_rates)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thresholds, avg_delays, 'o-', color='steelblue')
axes[0].set(xlabel='CUSUM threshold h', ylabel='Average detection delay (timesteps)',
            title='Detection delay vs. threshold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, false_alarm_rates, 's-', color='crimson')
axes[1].set(xlabel='CUSUM threshold h', ylabel='False-alarm rate',
            title='False-alarm rate vs. threshold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('CUSUM threshold sensitivity (200 trials each)', fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Lower threshold → faster detection, more false alarms.")
print("  Higher threshold → fewer false alarms, slower detection.")
print("  Choose h by deciding which error costs more in your application.")

## 7 — Key takeaways

| Detector | Strengths | Weaknesses |
|---|---|---|
| **CUSUM** | Simple, fast, one score to watch | Assumes you know the baseline; only detects upward shifts |
| **BOCD** | Full probability output; handles uncertainty naturally | Slower (O(T²) memory/compute); needs a good prior |
| **Window KL** | No assumptions about baseline; works on raw counts | Detection is always delayed by window_size; sensitive to window choice |

### What's next?

In **Phase 3** we make the noise model more realistic:
- **Dephasing noise** (relevant to neutral-atom platforms): noise that scrambles
  phase without directly flipping bits.
- **Correlated (non-Markovian) noise**: noise whose strength at time t depends
  on what happened at t-1 — the key model studied by Giarmatzi / Tonekaboni.

We'll see that correlated noise is *harder* to detect, and that adding
autocorrelation-based features to the input helps the detectors.

> **Your CUSUM detector caught the changepoint within a few timesteps of the
> true shift — that's exactly what real-time quantum device monitoring needs.**